## FORECASTING MODEL 

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Prepare Data
train_df = pd.read_csv('Advertising.csv')
train_df.columns = train_df.columns.str.lower()
X = train_df[['tv', 'radio', 'newspaper']]
y = train_df['revenue']

# 2. Train-Test Split (Standard 80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Fit the Perfected Model
forecaster = Ridge(alpha=1.0)
forecaster.fit(X_train, y_train)

# 4. Calculate Evaluation Metrics
preds = forecaster.predict(X_test)
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

# 5. 10-Fold Cross-Validation (The "Gold Standard" for Stability)
cv_scores = cross_val_score(forecaster, X, y, cv=10, scoring='r2')

# 6. Display the Professional Report
print(f"Accuracy (R2 Score):     {r2:.4f}")
print(f"Mean Absolute Error:     ${mae:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")

# DNA Ratios
total_avg_spend = train_df[['tv', 'radio', 'newspaper']].sum().sum()
spend_ratios = (train_df[['tv', 'radio', 'newspaper']].sum() / total_avg_spend).to_dict()



Accuracy (R2 Score):     0.8994
Mean Absolute Error:     $1.4608
Root Mean Squared Error: 1.7816


In [2]:
import os
import joblib 

try:
    # 1. Checking if model is fitted
    if not hasattr(forecaster, "intercept_"):
        raise ValueError("The 'forecaster' model has not been trained yet!")

    # 2. Checking for MAE variable
    if 'mae_margin' not in locals():
        # Fallback calculation if variable is missing
        from sklearn.metrics import mean_absolute_error
        mae_margin = mean_absolute_error(y, forecaster.predict(X))

    # 3. Create directory
    os.makedirs("models", exist_ok=True)

    # 4. Save Artifacts
    joblib.dump(forecaster, "models/forecaster.pkl")
    joblib.dump(spend_ratios, "models/spend_ratios.pkl")

    # 5. Create and Save Config
    config = {
        "intercept": float(forecaster.intercept_),
        "mae_margin": float(mae_margin),
        "training_r2": float(forecaster.score(X, y))
    }
    joblib.dump(config, "models/model_config.pkl")

    print(f"✅ Success! Baseline Organic Revenue: ${forecaster.intercept_:.2f}")
    print(f"✅ Files saved in: {os.path.abspath('models')}")

except NameError as e:
    print(f"❌ Error: A required variable is missing: {e}")
except Exception as e:
    print(f"❌ Unexpected Error: {e}")

✅ Success! Baseline Organic Revenue: $2.98
✅ Files saved in: c:\Users\Lakshita Chawla\OneDrive\ドキュメント\insightstream\notebooks\models


In [3]:
def process_any_dataset(df):
    """
    Automatically maps any dataframe to 'Marketing Spend' and 'Revenue'.
    """
    df.columns = df.columns.str.lower()
    
    # Expanded Keyword Lists for better accuracy
    rev_keywords = ['sales', 'revenue', 'income', 'credit', 'inward', 'upi', 'amount']
    mkt_keywords = ['marketing', 'ads', 'facebook', 'google', 'promotion', 'advertising', 'spend', 'newspaper', 'tv']
    
    # 1. Find the Value/Amount Column
    amt_col = next((c for c in df.columns if any(k in c for k in ['amount', 'value', 'price', 'total'])), None)
    
    # 2. Find the Category/Description Column
    cat_col = next((c for c in df.columns if any(k in c for k in ['category', 'type', 'description'])), None)
    
    if not amt_col or not cat_col:
        return "Error: Could not identify amount or category columns."

    # 3. Calculate Totals using Fuzzy Matching
    total_revenue = df[df[cat_col].str.contains('|'.join(rev_keywords), case=False, na=False)][amt_col].abs().sum()
    total_marketing = df[df[cat_col].str.contains('|'.join(mkt_keywords), case=False, na=False)][amt_col].abs().sum()
    
    # Calculate Efficiency (Handle 0 spend to prevent division error)
    efficiency = total_revenue / total_marketing if total_marketing > 0 else 0
    
    return {
        "identified_revenue": total_revenue,
        "identified_marketing": total_marketing,
        "efficiency": efficiency
    }

# Test on demo_data.csv
demo_df = pd.read_csv('demo_data.csv')
biz_state = process_any_dataset(demo_df)
print("Extracted Business State:", biz_state)

Extracted Business State: {'identified_revenue': 723100.0, 'identified_marketing': 45000.0, 'efficiency': 16.06888888888889}


In [4]:
def universal_simulator(new_total_marketing_spend, business_state):
    """
    Predicts revenue with high accuracy by scaling the 'Organic Baseline' 
    and 'Marketing Impact' separately.
    """
    model = joblib.load("models/forecaster.pkl")
    ratios = joblib.load("models/spend_ratios.pkl")
    config = joblib.load("models/model_config.pkl")
    
    # 1. Virtual Split based on Advertising DNA
    sim_input = pd.DataFrame([[
        new_total_marketing_spend * ratios['tv'],
        new_total_marketing_spend * ratios['radio'],
        new_total_marketing_spend * ratios['newspaper']
    ]], columns=['tv', 'radio', 'newspaper'])
    
    # 2. Predict using the model (this includes the model's inherent intercept)
    raw_prediction = model.predict(sim_input)[0]
    
    # 3. Scaling Logic (The 'Realism' Engine)
    # training_avg_eff is the ROI baseline of the Advertising dataset
    training_avg_eff = 0.07 
    scale_factor = (business_state['efficiency'] / training_avg_eff) if business_state['efficiency'] > 0 else 1.0
    
    # Scale the total prediction to match the user's business magnitude
    final_forecast = raw_prediction * scale_factor
    margin = config['mae_margin'] * scale_factor 
    
    return {
        "Projected Revenue": f"${final_forecast:,.2f}",
        "Range": f"${(final_forecast - margin):,.2f} - ${(final_forecast + margin):,.2f}",
        "ROI": f"{((final_forecast - new_total_marketing_spend) / new_total_marketing_spend) * 100:.2f}%" if new_total_marketing_spend > 0 else "0%",
        "Confidence": "95%"
    }

# Testing the refined logic
biz_state = process_any_dataset(demo_df) 
print(universal_simulator(50000, biz_state))

{'Projected Revenue': '$632,848.33', 'Range': '$632,561.18 - $633,135.49', 'ROI': '1165.70%', 'Confidence': '95%'}


## AUDITING MODEL

In [5]:
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Loading Brand Data for training
audit_train = pd.read_csv("Brand_Sales_AdSpend_Data.csv")
audit_train = audit_train.dropna(subset=["Total Ad Spend", "Net Sales"])

X_audit = pd.DataFrame({
    "marketing_spend": audit_train["Total Ad Spend"].abs(),
    "revenue": audit_train["Net Sales"].abs(),
    "is_duplicate": 0
})

# Production Pipeline
auditor_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("clf", EllipticEnvelope(contamination=0.03, random_state=42))
])

auditor_pipe.fit(X_audit)
joblib.dump(auditor_pipe, "models/auditor.pkl")
print("✅ Auditor model (auditor.pkl) saved successfully.")

C:\Users\Lakshita Chawla\AppData\Roaming\Python\Python312\site-packages\sklearn\covariance\_robust_covariance.py:749: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(


✅ Auditor model (auditor.pkl) saved successfully.


In [6]:
import pandas as pd
import numpy as np
import joblib

pipe = joblib.load("models/auditor.pkl")
demo_df = pd.read_csv("demo_data.csv")
demo_df = demo_df.dropna(subset=["amount"])

# Feature engineering 
demo_df["is_mkt"] = demo_df["category"].str.contains(
    "Marketing|Ads", case=False, na=False
).astype(int)

X_demo = pd.DataFrame({
    "marketing_spend": np.where(
        demo_df["is_mkt"] == 1,
        demo_df["amount"].abs(),
        0
    ),
    "revenue": demo_df["amount"].abs(),
    "is_duplicate": demo_df.duplicated().astype(int)
})

# Predicingt anomalies
demo_df["anomaly_flag"] = pipe.predict(X_demo)

# Converting to user-friendly label
demo_df["anomaly_label"] = demo_df["anomaly_flag"].map({
    1: "Normal",
    -1: "Anomaly"
})

# Displaying detected anomalies
anomalies = demo_df[demo_df["anomaly_flag"] == -1]

print("Total Transactions:", len(demo_df))
print("Anomalies Detected:", len(anomalies))

anomalies[[
    "date",
    "description",
    "category",
    "quantity",
    "amount",
    "anomaly_label"
]].head(10)


Total Transactions: 50
Anomalies Detected: 5


,date,description,category,quantity,amount,anomaly_label
17,2025-09-30,Salary Payment,Payroll,5,150000.0,Anomaly
24,2025-10-25,ANOMALY: Massive Refund,Refunds,1,200000.0,Anomaly
34,2025-11-30,Salary Payment,Payroll,5,150000.0,Anomaly
35,2025-12-01,Year End Sale,Sales,50,225000.0,Anomaly
47,2025-12-29,ANOMALY: High Value Gift,Marketing,1,120000.0,Anomaly


## ANOMALY DETECTION MODEL TOURNAMENT 

In [7]:
# import pandas as pd
# import numpy as np
# import joblib
# import time

# from sklearn.ensemble import IsolationForest
# from sklearn.neighbors import LocalOutlierFactor
# from sklearn.covariance import EllipticEnvelope
# from sklearn.svm import OneClassSVM

# from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer

# # =====================================================
# # CONFIG
# # =====================================================

# MIN_ANOM = 3
# MAX_ANOM = 12
# STABILITY_RUNS = 5
# NOISE_STD = 0.03     # 3% perturbation

# # =====================================================
# # LOAD DATA
# # =====================================================

# train_df = pd.read_csv("Brand_Sales_AdSpend_Data.csv")
# train_df = train_df.dropna(subset=["Total Ad Spend", "Net Sales"])

# X_train = pd.DataFrame({
#     "marketing_spend": train_df["Total Ad Spend"].abs(),
#     "revenue": train_df["Net Sales"].abs(),
#     "is_duplicate": 0
# })

# demo_df = pd.read_csv("demo_data.csv")
# demo_df = demo_df.dropna(subset=["amount"])

# demo_df["is_mkt"] = demo_df["category"].str.contains(
#     "Marketing|Ads", case=False, na=False
# ).astype(int)

# X_test = pd.DataFrame({
#     "marketing_spend": np.where(demo_df["is_mkt"] == 1, demo_df["amount"].abs(), 0),
#     "revenue": demo_df["amount"].abs(),
#     "is_duplicate": demo_df.duplicated().astype(int)
# })

# # =====================================================
# # ROBUST PIPELINES
# # =====================================================

# pipelines = {
#     "robust": [
#         ("imputer", SimpleImputer(strategy="median")),
#         ("scaler", RobustScaler())
#     ],
#     "log_robust": [
#         ("log", FunctionTransformer(np.log1p)),
#         ("imputer", SimpleImputer(strategy="median")),
#         ("scaler", RobustScaler())
#     ]
# }

# # =====================================================
# # MODEL SEARCH SPACE (SAFE + ROBUST ONLY)
# # =====================================================

# models = [
#     ("IF", IsolationForest(contamination=c, n_estimators=n, random_state=42))
#     for c in [0.03, 0.05, 0.07]
#     for n in [200, 300, 500]
# ] + [
#     ("LOF", LocalOutlierFactor(n_neighbors=k, contamination=c, novelty=True))
#     for k in [15, 20, 25]
#     for c in [0.03, 0.05]
# ] + [
#     ("EE", EllipticEnvelope(contamination=c, support_fraction=sf))
#     for c in [0.03, 0.05]
#     for sf in [0.75, 0.9]
# ]

# # =====================================================
# # STABILITY FUNCTION
# # =====================================================

# def stability_score(pipe, X):
#     preds = []
#     for _ in range(STABILITY_RUNS):
#         noise = np.random.normal(0, NOISE_STD, X.shape)
#         X_noisy = X * (1 + noise)
#         preds.append(pipe.predict(X_noisy))

#     agree = np.mean([
#         np.mean(preds[i] == preds[j])
#         for i in range(len(preds))
#         for j in range(i + 1, len(preds))
#     ])
#     return agree

# # =====================================================
# # RUN ROBUSTNESS TOURNAMENT
# # =====================================================

# leaderboard = []
# model_store = {}
# model_id = 1

# print(f"{'ID':<4} {'Model':<4} {'Pipe':<10} {'Anom':<6} {'Stability':<10} {'Verdict'}")
# print("-" * 70)

# for pipe_name, steps in pipelines.items():
#     for model_name, model in models:

#         pipe = Pipeline(steps + [("clf", model)])

#         try:
#             pipe.fit(X_train)
#             preds = pipe.predict(X_test)
#             count = np.sum(preds == -1)

#             if count < MIN_ANOM or count > MAX_ANOM:
#                 continue

#             stab = stability_score(pipe, X_test)

#             verdict = "🔥 ROBUST" if stab >= 0.85 else "⚠️ UNSTABLE"

#             leaderboard.append({
#                 "id": model_id,
#                 "model": model_name,
#                 "pipeline": pipe_name,
#                 "anomalies": count,
#                 "stability": stab,
#                 "params": pipe.named_steps["clf"].get_params()
#             })

#             model_store[model_id] = pipe

#             print(f"{model_id:<4} {model_name:<4} {pipe_name:<10} {count:<6} {stab:<10.3f} {verdict}")

#             model_id += 1

#         except Exception:
#             continue

# # =====================================================
# # PICK BEST MODEL AUTOMATICALLY
# # =====================================================

# df = pd.DataFrame(leaderboard)
# best_row = df.sort_values(
#     by=["stability", "anomalies"],
#     ascending=[False, True]
# ).iloc[0]

# BEST_ID = best_row["id"]
# best_model = model_store[BEST_ID]

# import json

# model_report = {
#     "model_type": best_row["model"],
#     "pipeline": best_row["pipeline"],
#     "anomaly_count": int(best_row["anomalies"]),
#     "stability": float(best_row["stability"]),
#     "hyperparameters": best_row["params"]
# }

# with open("best_model_config.json", "w") as f:
#     json.dump(model_report, f, indent=4)

# print("\n🏆 BEST ROBUST MODEL SELECTED")
# print(best_row)

# # =====================================================
# # SAVE FINAL MODEL
# # =====================================================

# joblib.dump(best_model, "auditor.pkl")
# print("\n✅ Saved as auditor.pkl")
